# 3 — Choosing the reference

**Same requirements as [Tutorial 2](02_subtype_markers): SCimilarity `model_v1.1`, a GPU, an
atlas. Runs in about a minute.**

This is the notebook that explains why RECAST is not a differential-expression test with extra
steps.

A DE test has one implicit question built into it: *this group versus everything else in the
object*. RECAST makes that comparison an explicit argument. Below, one population — Temra
cytotoxic T cells — is held completely fixed while the reference is changed four times. The
cells, the encoder, the gene space, and every other argument stay identical. Only the biological
alternative changes.

The panels that come back are not four noisy versions of one answer. They are answers to four
different questions, and the top of the list changes accordingly.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
import recast
from recast.encoders import SCimilarityEncoder
from scimilarity.utils import align_dataset

MODEL = "/data/gli9/Jian/sc_age_clock/models/scimilarity_model/model_v1.1"
H5AD  = "/data/gli9/test_sig/scattr_benchmark/phase2/dominguez_conde_fullgene.h5ad"

TARGET = "Tem/Temra cytotoxic T cells"
CYTO   = [TARGET, "Tcm/Naive cytotoxic T cells", "Tem/Trm cytotoxic T cells", "gdT"]


go   = pd.read_csv(f"{MODEL}/gene_order.tsv", header=None)[0].tolist()
full = align_dataset(ad.read_h5ad(H5AD), go)          # whole atlas, raw counts, aligned
enc  = SCimilarityEncoder(MODEL, device="cuda", normalize=True)

lin = full[full.obs["label"].isin(CYTO)].copy()       # the cytotoxic T lineage only
lin.obs["label"] = lin.obs["label"].cat.remove_unused_categories()

print("atlas:", full.shape, "| lineage subset:", lin.shape)
print(full.obs["label"].value_counts().to_string())

atlas: (25980, 28231) | lineage subset: (3750, 28231)
label
Tcm/Naive helper T cells       6884
Classical monocytes            5884
Tem/Effector helper T cells    4416
CD16+ NK cells                 3909
Tem/Temra cytotoxic T cells    1759
Tcm/Naive cytotoxic T cells     957
Regulatory T cells              722
Tem/Trm cytotoxic T cells       618
gdT                             416
Non-classical monocytes         213
Cycling immune mix              202


## Four questions about one population

| # | Reference | The question it asks |
|---|---|---|
| 1 | `"siblings"` on the lineage subset | What makes Temra different from the *other cytotoxic T subtypes*? |
| 2 | `"rest"` on the whole atlas | What makes Temra different from *every other immune cell here*? |
| 3 | `["Tcm/Naive cytotoxic T cells"]` | What makes Temra different from *naive* CD8 T cells? |
| 4 | a group we build ourselves | What makes Temra different from the *other effector* populations? |

Rows 1 and 2 differ only in which object is passed — `reference="siblings"` and `"rest"` are the
same rule (*everything that is not the target*), so the object boundary carries the meaning. Rows
3 and 4 name the reference explicitly: `reference` accepts a list of labels, and those labels can
be any grouping you put in `.obs`, not just your published annotation.

In [2]:
# a grouping that exists nowhere in the annotation: pool Trm + gdT into one reference
lin.obs["grp"] = pd.Categorical(np.where(
    lin.obs["label"] == TARGET, TARGET,
    np.where(lin.obs["label"].isin(["Tem/Trm cytotoxic T cells", "gdT"]), "other effectors", "naive")))
print(lin.obs["grp"].value_counts().to_string())

QUESTIONS = [
    ("vs siblings",        lin,  "label", "siblings"),
    ("vs rest of atlas",   full, "label", "rest"),
    ("vs naive only",      lin,  "label", ["Tcm/Naive cytotoxic T cells"]),
    ("vs other effectors", lin,  "grp",   ["other effectors"]),
]

panels, dprime = {}, {}
for name, obj, key, ref in QUESTIONS:
    r = recast.attribute(enc, obj, key, target=TARGET, reference=ref, device="cuda", qc="silent")
    panels[name] = r.top(TARGET, 15)
    dprime[name] = float(r.qc["dprime"].iloc[0])
    print(f"\n[{name}]  d' = {dprime[name]:.2f}")
    print(" ", panels[name])

grp
Tem/Temra cytotoxic T cells    1759
other effectors                1034
naive                           957



[vs siblings]  d' = 1.56
  ['NKG7', 'GNLY', 'KLRD1', 'GZMH', 'CTSW', 'CST7', 'GZMA', 'PRF1', 'GZMB', 'FGFBP2', 'FCGR3A', 'KLRK1', 'GZMM', 'CCL5', 'CCL4']



[vs rest of atlas]  d' = 3.28
  ['CD8A', 'CD8B', 'GZMH', 'KLRD1', 'NKG7', 'CCL5', 'LINC02446', 'CD3D', 'KLRK1', 'CD3G', 'GZMA', 'CD3E', 'CTSW', 'CST7', 'IL32']



[vs naive only]  d' = 6.80
  ['NKG7', 'CTSW', 'KLRD1', 'CST7', 'GNLY', 'GZMA', 'PRF1', 'GZMH', 'FCGR3A', 'KLRK1', 'GZMB', 'CCL5', 'GZMM', 'GZMK', 'CMC1']



[vs other effectors]  d' = 1.26
  ['GNLY', 'GZMH', 'FGFBP2', 'GZMB', 'CD52', 'KLRD1', 'NKG7', 'PRF1', 'CRIP1', 'CD3G', 'ZEB2', 'EMP3', 'IL2RG', 'S100A4', 'FLNA']


Read those four lists side by side — the differences are not subtle.

**vs siblings** returns the cytotoxic effector program: `NKG7`, `GNLY`, granzymes, `PRF1`,
`FGFBP2`. Everything in the reference is also a CD8 T cell, so nothing about being a T cell can
appear.

**vs rest of atlas** puts `CD8A` and `CD8B` first, with `CD3D`, `CD3G` and `CD3E` close behind —
the CD8 co-receptor and the pan-T-cell receptor complex. Not one of them appears anywhere in the
sibling panel, and that is exactly right:
against monocytes, B cells and NK cells, *being a CD8 T cell* is the most informative thing about
this population. Same cells, same encoder, one argument different, and the answer moves from a
subtype program to a lineage-identity program.

**vs naive only** is close to the sibling panel — which is itself the finding. Naive cells are
roughly half the sibling reference, so the pooled "siblings" contrast is largely being driven by
them. A pooled reference is weighted by whatever happens to be abundant in it.

**vs other effectors** is the hardest and most specific question: Temra against Trm and γδ, the
two populations that also kill. The granzymes survive, but `ZEB2`, `S100A4`, `EMP3` and `CRIP1`
move up — the terminal-differentiation program rather than the generic cytotoxic one.

## The same thing, as numbers

In [3]:
ks = list(panels)
ov = pd.DataFrame([[len(set(panels[a]) & set(panels[b])) for b in ks] for a in ks],
                  index=ks, columns=ks)
print("shared genes among each pair of top-15 panels:\n")
print(ov.to_string())
print("\nseparation of the contrast:")
print(pd.Series(dprime).round(2).to_string())

shared genes among each pair of top-15 panels:

                    vs siblings  vs rest of atlas  vs naive only  vs other effectors
vs siblings                  15                 8             13                   7
vs rest of atlas              8                15              8                   4
vs naive only                13                 8             15                   6
vs other effectors            7                 4              6                  15

separation of the contrast:
vs siblings           1.56
vs rest of atlas      3.28
vs naive only         6.80
vs other effectors    1.26


Two readings, both worth having.

**The panels.** The two extreme questions — "what makes this a T cell" and "what makes this a
*terminal* effector rather than another kind of effector" — share only a handful of their top 15.
Between those extremes the overlap grades smoothly. There is no single Temra marker panel to be
recovered; there is a panel per question.

**The `d'` column.** Separation tracks how similar the reference is to the target: naive cells are
easy to separate Temra from, the whole atlas is intermediate, and the other effector populations
are hardest. This is the point [Tutorial 2](02_subtype_markers) ended on, now shown deliberately
rather than observed after the fact — **`d'` is a property of the question, not a quality score
for your cluster.** A low `d'` on the "vs other effectors" contrast does not mean the annotation
is bad; it means you asked a genuinely fine-grained question.

:::{admonition} Which reference is the right one?
:class: tip

The one that matches the claim you intend to make.

- Writing "markers of subtype X **within** lineage L"? Subset to L and use `"siblings"`. This is
  what the manuscript's subtype benchmark does.
- Writing "markers that identify cell type X in a mixed sample"? Use `"rest"` on the full object.
- Testing a specific hypothesis ("X differs from Y")? Name Y with a list. This is also the honest
  option when your object contains populations that are irrelevant to the comparison you mean.
- Unsure whether your annotation's boundaries are the right ones? Build a grouping in `.obs` and
  pass that — the reference is a set of cells, not a category in your metadata.

What you should not do is run one reference, get a panel, and describe it as though it were the
other reference's answer. That is the failure this notebook exists to make visible.
:::

## Where to go next

- [Tutorial 4](04_contrast_qc) — every contrast here was healthy. What a broken one looks like,
  and why a broken contrast still returns a confident-looking panel.
- [Tutorial 5](05_scoring_and_transfer) — freeze one of these panels and score it on cells that
  had no part in selecting it.